# Semantic Search for Lenta.ru News
## Business Task

The user enters a **search query** (“inflation”, “ruble exchange rate”, “Champions League final”), and the system returns relevant articles, even if the exact query words do not appear in the text.

**Goal**: compare a lexical baseline (TF-IDF) with a semantic approach (Word2Vec + cosine similarity) in a production-style setup: configuration, classes, train/index split, metrics, and artifacts.

## Dataset

[zloelias/lenta-ru](https://huggingface.co/datasets/zloelias/lenta-ru) — a Russian news dataset containing categories (topic), headlines, and article texts. It is suitable for search tasks and topic-based validation.

## Pipeline

```text
Loading → EDA → preprocessing → split (train/val/test)
    → indexing (TF-IDF | W2V document vectors) using train only
    → search API → validation (MRR, Recall@k, Topic-Precision@k)
    → method comparison → artifact saving
```

In [51]:
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
import json

import re

import nltk
from nltk.corpus import stopwords as nltk_stopwords
from razdel import tokenize as razdel_tokenize

import gensim
from gensim.models import Word2Vec

from datasets import load_dataset
from dataclasses import asdict, dataclass, field

import sklearn
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 4)

In [52]:
@dataclass(frozen=True)
class SemanticSearchConfig:
    dataset_id: str = "zloelias/lenta-ru"
    random_state: int = 42
    corpus_sample: int = 8_000

    min_tokens_per_doc: int = 12
    min_word_len: int = 2

    title_boost: int = 3
    tfidf_max_features: int = 20_000
    embed_dim: int = 128

    w2v_window: int = 5
    w2v_min_count: int = 3
    w2v_epochs: int = 15

    lsa_components: int = 200

    search_top_k: int = 10

    hybrid_candidates: int = 50
    hybrid_alpha: float = 0.85

    title_query_tokens: int = 5

    artifacts_dir: Path = field(default_factory=lambda: Path("artifacts"))

    def to_dict(self) -> dict:
        d = asdict(self)
        d["artifacts_dir"] = str(self.artifacts_dir)
        return d

In [53]:
CFG = SemanticSearchConfig()

CFG.artifacts_dir.mkdir(parents=True, exist_ok=True)
np.random.seed(CFG.random_state)

In [54]:
raw = load_dataset(CFG.dataset_id, split="train")
df_all = raw.to_pandas()
df = (
    df_all.groupby("topic", group_keys=False)
          .sample(frac=CFG.corpus_sample / len(df_all), random_state=CFG.random_state)
          .reset_index(drop=True)
)

In [60]:
df["doc_id"] = df.index.astype(str)
df['title'] = df['title'].fillna('').str.strip()
df['text'] = df['text'].fillna('').str.strip()
df["full_text"] = (df["title"] + " " + df["text"]).str.strip()

In [61]:
print("Статей:", len(df))
print(df["topic"].value_counts())
df["full_text"][0]

Статей: 7999
topic
Экономика          2462
Спорт              1997
Культура           1669
Наука и техника    1641
Бизнес              230
Name: count, dtype: int64


'Россияне стали покупать больше мотоциклов Продажи новой мототехники в России по итогам прошлого года выросли в 1,5 раза, составив 30,2 тысячи единиц. Об этом сообщается в пресс-релизе аналитического агентства «Автостат», поступившем в редакцию «Ленты.ру». Больше всего в 2014-м было продано техники российской компании Irbis — 4,8 тысячи единиц, за ней следуют российско-китайские Stels — 2,9 тысячи — и отечественный Racer — 2,8 тысячи штук. На четвертой и пятой позициях оказались японские Yamaha с 2,5 тысячи проданных единиц техники и Honda — 2 тысячи штук. В январе-марте спрос на мотоциклы упал на 19,6 процента: было продано 2100 штук. Из них больше всего техники немецкой компании BMW (267 единиц), российской Irbis (219 штук) и японской Yamaha (206 штук). Рынок мотоциклов, мотороллеров и другой двухколесной техники в 2014 году вырос на фоне затяжного кризиса на авторынке. В 2014-м, по данным Ассоциации европейского бизнеса, продажи легковых и легких коммерческих автомобилей в РФ упали 